Nama : Dimas Prih P J<br>
NIM : 240401010264<br>
Kelas : IF403<br>

#1. Generate & Eksplorasi Dataset Transaksi
- Buat dataset transaksi sintetis dengan pola pembelian tersembunyi, lalu eksplorasi frekuensi tiap produk.

In [14]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
  if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


Program menghasilkan 50 baris data transaksi. Masing-masing baris merepresentasikan satu struk belanja atau satu kali kunjungan pembeli ke toko yang berisi beberapa produk.

#2. One-Hot Encoding Transaksi
- Ubah daftar transaksi menjadi tabel one-hot encoding menggunakan TransactionEncoder.

In [15]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


Setiap kolom merepresentasikan nama produk secara berurutan alfabetis (dari Gula hingga Telur), sedangkan setiap baris merepresentasikan transaksi belanja (indeks 0 hingga 4). Nilai True menunjukkan bahwa produk tersebut dibeli pada transaksi tersebut, sedangkan nilai False berarti produk tidak dibeli.

#3. Cari Frequent Itemset dengan Apriori
- Jalankan Apriori dengan beberapa nilai min_support, amati bagaimana jumlah itemset yang ditemukan berubah.

In [16]:
from mlxtend.frequent_patterns import apriori
import warnings

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


#4. Bentuk & Saring Aturan Asosiasi
- Bentuk aturan asosiasi, saring dengan min_confidence dan min_lift, lalu urutkan berdasarkan Lift tertinggi.

In [17]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence',
                           min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))


         antecedents consequents  support  confidence      lift
9        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
15  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
12      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
8      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
13   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


- Aturan mana yang paling kuat (Lift tertinggi)?
- Jawab : (Keju, Teh)->(Telur). Nilai Metrik:
  - Support: 0.12 (muncul bersamaan di 12% transaksi)
  - Confidence: 0.857 atau 85.7% (pembeli Keju dan Teh memiliki kemungkinan 85.7% juga membeli Telur)
  - Lift: 2.38 (nilai tertinggi di antara semua aturan, menunjukkan hubungan yang kuat dan positif antar produk).

- Apakah masuk akal secara bisnis?
- Jawab: kombinasi sarapan seperti (Keju, Teh) -> (Telur) atau seperti (Roti) -> (Selai) masuk akal. Barang-barang ini adalah produk fmcg / bahan sarapan yang sering dibeli bersamaan dalam satu keranjang belanja.

#5. Rekomender Sederhana dengan Content-Based Filtering
- Bangun katalog produk dengan kategori, lalu buat rekomendasi produk serupa menggunakan cosine similarity atas kategori (one-hot).

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
  idx = katalog.index[katalog['produk'] == nama_produk][0]
  skor = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]
  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


Sistem merekomendasikan tiga produk teratas yang memiliki tingkat kemiripan fitur kategori paling tinggi dengan Roti, di mana Selai, Sereal, dan Susu berada pada urutan teratas daftar kemiripan tersebut.

#6. Bandingkan Kedua Pendekatan
- Bandingkan rekomendasi dari aturan asosiasi (Langkah 4) dengan rekomendasi Content- Based (Langkah 5) untuk produk yang sama.

In [19]:
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Association Rules menghasilkan produk Selai (berdasarkan kebiasaan pola transaksi nyata dengan nilai lift masing-masing 1.92 dan 1.32), sementara Content-Based Filtering merekomendasikan daftar ['Selai', 'Sereal', 'Susu'] berdasarkan kemiripan kategori produk (Bakery).

- Apakah kedua pendekatan memberi rekomendasi yang konsisten?
- Jawab : Kedua pendekatan tersebut memberikan rekomendasi yang cukup konsisten, terutama pada produk pelengkap utama. Hal ini dilihat dari rekomendasi Selai yang sama sama muncul sebagai hasil teratas pada metode Association Rules maupun Content-Based Filtering untuk produk target Roti.
- Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?
- Jawab :
  - Gunakan Association Rules ketika menggunakan data transaksi historis pelanggan yang berlimpah.
  - Gunakan Content-Based Filtering ketika ada produk baru di katalog yang belum memiliki riwayat transaksi atau data pembelian sama sekali.
  - Gabungkan Keduanya (Sistem Hybrid) ketika ingin membangun sistem rekomendasi yang komprehensif (misal e-commerce besar). Pendekatan hybrid menggabungkan kelebihan keduanya, menggunakan Content-Based untuk menangani produk baru, serta menggunakan Association Rules untuk memanfaatkan tren keranjang belanja yang akurat.

#Yang Saya Pelajari
Pada pertemuan ini, saya mempelajari cara membangun sistem rekomendasi dasar menggunakan algoritma Association Rules (Apriori) untuk menganalisis pola pembelian keranjang belanja dan Content-Based Filtering berbasis cosine similarity untuk mengelompokkan produk berdasarkan kategori.
#Temuan Utama
Metode asosiasi berhasil mendeteksi pola produk yang sering dibeli bersamaan, baik secara alami maupun dari data buatan seperti relasi Roti dan Selai. Pendekatan berbasis konten sukses merekomendasikan item sejenis berdasarkan kategori produknya.
#Keterbatasan / Pertanyaan
Bagaimana menangani produk baru tanpa riwayat transaksi.